In [1]:
keywords = [
    "stock", "price", "shares", "stake", "deal", "merger", "acquisition", "dividend",
    "forecast", "earnings", "profit", "revenue", "sales", "cost", "expenses", "demand",
    "capacity", "liquidity", "loan", "NPL", "market share", "customer", "growth"
]


In [4]:
import json
import re

# Load raw sentences (list of dicts with "content" or "sentence")
with open("sentences_only _Copy.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# util: simple whitespace tokenizer (you can use a better tokenizer if you want)
def simple_tokenize(text):
    # keep punctuation as separate tokens roughly
    return re.findall(r"\w+|[^\s\w]", text)

def label_tokens(tokens, keywords):
    labels = ["O"] * len(tokens)
    text_lower = " ".join(tokens).lower()
    # naive matching: find occurrences of each keyword in string form of tokens
    for kw in keywords:
        kw_tokens = kw.split()
        # join tokens and search for contiguous sequence
        for i in range(len(tokens) - len(kw_tokens) + 1):
            segment = " ".join(tokens[i:i+len(kw_tokens)]).lower()
            if segment == kw.lower():
                labels[i] = "B-ASP"
                for j in range(1, len(kw_tokens)):
                    labels[i+j] = "I-ASP"
    return labels

auto_labeled = []
for item in raw:
    sentence = item.get("content") or item.get("sentence")
    tokens = simple_tokenize(sentence)
    labels = label_tokens(tokens, keywords)
    auto_labeled.append({"sentence": sentence, "tokens": tokens, "labels": labels})

# Save weakly labeled output to review
with open("auto_labeled_weak.json", "w", encoding="utf-8") as f:
    json.dump(auto_labeled, f, indent=2, ensure_ascii=False)
